[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/9_evaluation.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

# Automatic evaluation

Automatic evaluation uses algorithmic metrics ato measure system behavior without (or before) human raters. It is fast and useful for development and large-scale comparisons, but it can miss subjective aspects (fluency, naturalness) and complex user goals.

## Components
Useful to evaluate individual modules in isolation (e.g., NLU, DM, NLG) to understand whether a certain prompt or a specific technique (e.g., zero-shot vs. few-shot) is preferred for a given task.

- Typical metrics:
    - NLU: intent accuracy, slot F1
    - DM: action classification F1
    - NLG: BLEU, ROUGE

- Pros:
    - Clear attribution of errors to a specific component.
    - Faster iteration and targeted debugging.
- Cons:
    - Doesn't capture error propagation between modules.
    - Metrics may not correlate with user judgment.

## Pipeline

Although Human Evaluation is correct and proper way to assess the performance of our system, we can get a rough estimate by combining the results from the intermediate components.

ATTENTION: since you won't have to fine-tune any model for your project, you are expected to test different prompts/techniques/models to understand their impact on the system performance.

## Data Generation 

Generating synthetic dialogue data is particularly useful to evaluate the capabilities of your model. Two common patterns are:

- String injection / templates: design deterministic templates (with slot placeholders) and systematically fill them to generate many examples. This is explainable, fast, and useful for covering combinatorial slot spaces.

- LLM-based generation (few-shot / prompt-based): craft a prompt (optionally provide few-shot examples) and ask the model to generate user turns, system turns, or fully annotated examples (JSON, CSV, or plain text). This is flexible but requires manual validation.

Best practices:
- Alway validate LLM' outputs.
- Use a mix of both techniques: templates for coverage and LLMs for natural variation.

### Example (NLU)

In [3]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.86it/s]


In [6]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

task_prompt = """You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}
Generate a dialogue of six turns (three user utterances and three system responses) that reflects the given state.
Do not add, omit, or change any information from the dialogue state.
At each turn system turn, provide the dialogue state up to that point.

Output the result in the following JSON format:
[
    {
        "speaker": "USER" | "SYSTEM",
        "utterance": "...",
        # Only for SYSTEM turns
        "dialogue_state": {
            "intent": "...",
            "slots": {
                "slot": "value"
            }
        }
    },
    ...
]

Example:
Input:
{
    "intent": "pizza_ordering",
    "slots": {
        "pizza_type": "margherita",
        "pizza_size": "small",
        "pizza_count": "2"
    }
}
Output:
[
    {
        "speaker": "USER",
        "utterance": "I would like to order two pizzas."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Sure! You would like to order two pizzas. What type pizzas would you like?",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": null,
                "pizza_size": "small",
                "pizza_count": null
            }
    },
    {
        "speaker": "USER",
        "utterance": "Make a margherita pizza, please."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Got it! How many margherita pizzas would you like?",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": "margherita",
                "pizza_size": "small",
                "pizza_count": null
            }
    },
    {
        "speaker": "USER",
        "utterance": "Just two, thanks."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Great! To confirm, you would like to order two small margherita pizzas.",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": "margherita",
                "pizza_size": "small",
                "pizza_count": "2"
            }
    }
]
"""

In [ ]:
# 1. Dialogue Generation using LLMs

ds = """{
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "salamino",
    "pizza_count": "1"
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(ds, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs, max_new_tokens=512).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, ds, content)


### Conversation

**System:** You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}
Generate a dialogue of six turns (three user utterances and three system responses) that reflects the given state.
Do not add, omit, or change any information from the dialogue state.
At each turn system turn, provide the dialogue state up to that point.

Output the result in the following JSON format:
[
    {
        "speaker": "USER" | "SYSTEM",
        "utterance": "...",
        # Only for SYSTEM turns
        "dialogue_state": {
            "intent": "...",
            "slots": {
                "slot": "value"
            }
        }
    },
    ...
]

Example:
Input:
{
    "intent": "pizza_ordering",
    "slots": {
        "pizza_type": "margherita",
        "pizza_size": "small",
        "pizza_count": "2"
    }
}
Output:
[
    {
        "speaker": "USER",
        "utterance": "I would like to order two pizzas."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Sure! You would like to order two pizzas. What type pizzas would you like?",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": null,
                "pizza_size": "small",
                "pizza_count": null
            }
    },
    {
        "speaker": "USER",
        "utterance": "Make a margherita pizza, please."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Got it! How many margherita pizzas would you like?",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": "margherita",
                "pizza_size": "small",
                "pizza_count": null
            }
    },
    {
        "speaker": "USER",
        "utterance": "Just two, thanks."
    },
    {
        "speaker": "SYSTEM",
        "utterance": "Great! To confirm, you would like to order two small margherita pizzas.",
        "dialogue_state": {
            "intent": "pizza_ordering",
            "slots": {
                "pizza_type": "margherita",
                "pizza_size": "small",
                "pizza_count": "2"
            }
    }
]


**User:** {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "salamino",
    "pizza_count": "1"
  }
}

**Assistant:** [
  {
    "speaker": "USER",
    "utterance": "I'd like to order one salamino pizza in medium size."
  },
  {
    "speaker": "SYSTEM",
    "utterance": "Got it! You would like to order one salamino pizza in medium size. Is that correct?",
    "dialogue_state": {
      "intent": "pizza_ordering",
      "slots": {
        "pizza_size": "medium",
        "pizza_type": "salamino",
        "pizza_count": "1"
      }
    }
  },
  {
    "speaker": "USER",
    "utterance": "Yes, that's exactly what I wanted."
  },
  {
    "speaker": "SYSTEM",
    "utterance": "Great! To confirm, you would like to order one medium salamino pizza.",
    "dialogue_state": {
      "intent": "pizza_ordering",
      "slots": {
        "pizza_size": "medium",
        "pizza_type": "salamino",
        "pizza_count": "1"
      }
    }
  },
  {
    "speaker": "USER",
    "utterance": "Thank you for your help!"
  },
  {
    "speaker": "SYSTEM",
    "utterance": "You're welcome! Your order for one medium salamino pizza is confirmed.",
    "dialogue_state": {
      "intent": "pizza_ordering",
      "slots": {
        "pizza_size": "medium",
        "pizza_type": "salamino",
        "pizza_count": "1"
      }
    }
  }
]

In [15]:
import json

print(json.dumps(json.loads(content), indent=2))

[
  {
    "speaker": "USER",
    "utterance": "I'd like to order one salamino pizza in medium size."
  },
  {
    "speaker": "SYSTEM",
    "utterance": "Got it! You would like to order one salamino pizza in medium size. Is that correct?",
    "dialogue_state": {
      "intent": "pizza_ordering",
      "slots": {
        "pizza_size": "medium",
        "pizza_type": "salamino",
        "pizza_count": "1"
      }
    }
  },
  {
    "speaker": "USER",
    "utterance": "Yes, that's exactly what I wanted."
  },
  {
    "speaker": "SYSTEM",
    "utterance": "Great! To confirm, you would like to order one medium salamino pizza.",
    "dialogue_state": {
      "intent": "pizza_ordering",
      "slots": {
        "pizza_size": "medium",
        "pizza_type": "salamino",
        "pizza_count": "1"
      }
    }
  },
  {
    "speaker": "USER",
    "utterance": "Thank you for your help!"
  },
  {
    "speaker": "SYSTEM",
    "utterance": "You're welcome! Your order for one medium salamino pizza is

In [ ]:
# 2. String injection / templates
from itertools import product

template = "I would like to order {count} {size} {type} pizza{plural}"
pizza_types = ["margherita", "pepperoni", "veggie"]
pizza_sizes = [None, "small", "medium", "large"]
pizza_counts = [1, 2]

generated = []
for t, s, c in product(pizza_types, pizza_sizes, pizza_counts):
    plural = "s" if c > 1 else ""
    utt = template.format(count=c, size=s if s else "", type=t, plural=plural)
    example = {
        "utterance": " ".join(utt.split()),
        "intent": "pizza_ordering",
        "slots": {"pizza_type": t, "pizza_size": s, "pizza_count": c},
    }
    generated.append(example)

print("Generated examples:", len(generated))
print(generated[:4])

Generated examples: 24
[{'utterance': 'I would like to order 1 margherita pizza', 'intent': 'pizza_ordering', 'slots': {'pizza_type': 'margherita', 'pizza_size': None, 'pizza_count': 1}}, {'utterance': 'I would like to order 2 margherita pizzas', 'intent': 'pizza_ordering', 'slots': {'pizza_type': 'margherita', 'pizza_size': None, 'pizza_count': 2}}, {'utterance': 'I would like to order 1 small margherita pizza', 'intent': 'pizza_ordering', 'slots': {'pizza_type': 'margherita', 'pizza_size': 'small', 'pizza_count': 1}}, {'utterance': 'I would like to order 2 small margherita pizzas', 'intent': 'pizza_ordering', 'slots': {'pizza_type': 'margherita', 'pizza_size': 'small', 'pizza_count': 2}}]


<details>
    <summary> Can you think about a way to combine both?
    </summary>
    We can ask LLMs to generate a set of templates, validate them, and then use string injection.
</details>

In [8]:
task_prompt = """You are given the dialogue state for a task-oriented dialogue.
The dialogue state contains information about the user's intent and the slots:
{
    "intent": "...", 
    "slots": [slot1, slot2, ...]
}

Generate five templates for user utterances that reflect the given state.
For each template, use placeholders {slot1}, {slot2}, etc. to indicate where slot values should be inserted.
Ensure that the templates are diverse in structure and wording, while still accurately representing the intent and slots.

Do not add additional slots or intents.
Output the result in the following JSON format:
[
    "template1",
    "template2",
    "template3",
    "template4",
    "template5"
]
"""

In [9]:
# 3. Template generation with LLMs

ds = """{
  "intent": "pizza_ordering",
  "slots": ["pizza_size", "pizza_type", "pizza_count"]
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(ds, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs, max_new_tokens=512).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, ds, content)


### Conversation

**System:** You are given the dialogue state for a task-oriented dialogue.
The dialogue state contains information about the user's intent and the slots:
{
    "intent": "...", 
    "slots": [slot1, slot2, ...]
}

Generate five templates for user utterances that reflect the given state.
For each template, use placeholders {slot1}, {slot2}, etc. to indicate where slot values should be inserted.
Ensure that the templates are diverse in structure and wording, while still accurately representing the intent and slots.

Do not add additional slots or intents.
Output the result in the following JSON format:
[
    "template1",
    "template2",
    "template3",
    "template4",
    "template5"
]


**User:** {
  "intent": "pizza_ordering",
  "slots": ["pizza_size", "pizza_type", "pizza_count"]
}

**Assistant:** [
    "I'd like to order {pizza_count} pizzas of size {pizza_size}, specifically {pizza_type}.",
    "Can I get {pizza_count} {pizza_type} pizzas in {pizza_size} size?",
    "How about {pizza_count} {pizza_type} pizzas, each in {pizza_size}?",
    "Order me {pizza_count} of {pizza_type} in {pizza_size} please.",
    "I'm looking to order {pizza_count} {pizza_type} pizzas, and I want them to be {pizza_size}."
]

In [10]:
import json

print(json.dumps(json.loads(content), indent=2))

[
  "I'd like to order {pizza_count} pizzas of size {pizza_size}, specifically {pizza_type}.",
  "Can I get {pizza_count} {pizza_type} pizzas in {pizza_size} size?",
  "How about {pizza_count} {pizza_type} pizzas, each in {pizza_size}?",
  "Order me {pizza_count} of {pizza_type} in {pizza_size} please.",
  "I'm looking to order {pizza_count} {pizza_type} pizzas, and I want them to be {pizza_size}."
]


### Exercise (DM)

Create a prompt to generate data to evaluate the Dialogue Manager (DM).

Think about:
- what is the DM input
- its output
- which strategy could be the most effective

In [ ]:
# Put your code here

# Human Evaluation
From the definition on your slides, we can ensure the quality of crowdsourcing by following these guidelines:

1. Carefully define the experimental setup before starting the task
   - Privacy Assessment
   - Guidelines
   - The Task structure
   - User Interface

2. Monitor the progress and the result quality within the task
   - Crowd Qualification, Worker Sampling, Training
   - Continuous Quality Control
   - Provide Constructive feedback

3. Appropriate analysis of the data after the task
   - Agreement and accuracy of your crowd
   - Correct interpretation

## Guideline

The Guidelines must:
- provide the task description
- inform the crowd about the requirements
- present some examples and temples

For example:

*In this task, you are asked to interact with a system (reservation agent) that should help you (the user) booking a trip.*

*Specifically, you should ask the system to book a 5 days trip to new york. Additional information:*
- *you will be departing from rome*
- *you should book a three star hotel.*

*At the end of the conversation, you will be asked to answer a few question to assess the overall quality of your conversation with the system.*

*These questions include:*
- *Did the system follow your instructions and booking constraints (departure city, trip length, hotel category) and complete the task as requested?*
- *Did the system ask for missing information or clarifications when something was unclear?*
- *Were the system's responses natural, coherent, and free from contradictions or irrelevant details?*
- *Would you consider using the system again?*

*Answer them by providing a score from 1 to 5:*
1. *Completly Disagree*
2. *Disagree*
3. *Don't Know / Neutral*
4. *Agree*
5. *Completely Agree*

KEEP IN MIND THAT:
- examples are the preferred (and **suggested**) way to teach a user how to answer a given question (especially when faced with corner cases)
- sometimes the user may be asked to simply interact with the system, with no specific requirement (e.g., no information about the trip departure, duration, etc.) This is completely fine, but the results might only provide a broad evaluation of the system capabilities. If you want to tackle some corner cases, be more specific!

In [ ]:
# modify the previous guidelines and include examples 
# for each question to help the user better understand
# the annotation task

### Some Advices

- You can ask your friends/colleagues to help you crowdsourcing
- Before executing the whole experiment on 5/10 users, first perform some test on "beta testers" and ask for feedback (e.g., unclear or ambiguous information). Do NOT ask them to perform the task again!